# Paper 2 — Learned Routing in Heterogeneous LLM Cascades

This notebook is a **reproducibility-faithful analysis** of a single benchmark
run produced by `python -m python_core.scripts.run_experiment`. It does **not**
run its own experiments — that would re-introduce the silent-disagreement bug
between notebook and canonical artifact that we explicitly designed against in
Phase 3. Every table and figure below is derived from `manifest.json`,
`raw_stats.json`, and `pareto.csv` under a single `RESULTS_DIR`.

To regenerate the artifacts before re-running this notebook:

```bash
python -m python_core.scripts.run_experiment \
    --dataset tatsu-lab/alpaca_eval \
    --max-samples 200 --seeds 5 \
    --label "paper2-final"
```

The theoretical framing for the routing policies appears in `paper/theory.tex`.


In [ ]:
import json
import sys
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Point this at the directory produced by run_experiment.py.
# Pick the most recent run by default; override explicitly for reproducibility.
RESULTS_ROOT = Path('../results')
runs = sorted(RESULTS_ROOT.glob('*/')) if RESULTS_ROOT.exists() else []
if not runs:
    raise FileNotFoundError(
        f"No experiment runs found under {RESULTS_ROOT.resolve()}. "
        f"Generate one with:\n"
        f"  python -m python_core.scripts.run_experiment --max-samples 200 --seeds 5"
    )
RESULTS_DIR = runs[-1]
print(f"Using run: {RESULTS_DIR}")

MANIFEST = json.loads((RESULTS_DIR / 'manifest.json').read_text())
RAW_STATS = json.loads((RESULTS_DIR / 'raw_stats.json').read_text())
PARETO = pd.read_csv(RESULTS_DIR / 'pareto.csv')


## 1. Run Provenance

The reviewer-verifiable identity of this run. The `dataset.sha256` fixes the
inputs; `git_sha` + `git_dirty` fix the code; `package_versions` fix the
toolchain. Together these uniquely identify the experiment.


In [ ]:
print(f"Run label:      {MANIFEST.get('label', '(none)')}")
print(f"Timestamp (UTC): {MANIFEST['timestamp_utc']}")
print(f"Git SHA:        {MANIFEST['git_sha']} (dirty={MANIFEST['git_dirty']})")
print(f"Python:         {MANIFEST['python_version']}")
print(f"Dataset:        {MANIFEST['dataset']['name']} (split={MANIFEST['dataset']['split']})")
print(f"  Loaded:       {MANIFEST['dataset']['n_loaded']} prompts")
print(f"  SHA-256:      {MANIFEST['dataset']['sha256']}")
print(f"Seeds:          {MANIFEST['experiment']['seeds']}")
print(f"System mode:    {MANIFEST['experiment']['system_mode']}")
print(f"Engines:        {MANIFEST['engines']['label']}")
print()
print('Package versions:')
for k, v in MANIFEST['package_versions'].items():
    print(f'  {k:<25s} {v}')


## 2. Cost vs Quality Pareto Frontier

The headline figure. Each point is one router at the run's mean cost and mean
quality across the `n_seeds` seeds; error bars are 95% confidence intervals
computed from the sample standard deviation using the t-distribution (df=n−1).
The green dashed line connects the non-dominated set: routers on this frontier
are not strictly worse than any other router in both cost and quality
simultaneously. Routers off the frontier are dominated and would not be
recommended by a Pareto-rational decision-maker.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Frontier polyline.
frontier = PARETO[PARETO['on_frontier'] == 1].sort_values('cost_mean')
if len(frontier) >= 2:
    ax.plot(frontier['cost_mean'], frontier['quality_mean'],
            '--', color='tab:green', alpha=0.6, label='Pareto frontier')

for _, r in PARETO.iterrows():
    on_f = bool(r['on_frontier'])
    ax.errorbar(
        r['cost_mean'], r['quality_mean'],
        xerr=r['cost_ci'], yerr=r['quality_ci'],
        fmt='o' if on_f else 'x',
        color='tab:green' if on_f else 'tab:gray',
        markersize=8, capsize=3,
    )
    ax.annotate(r['router'], (r['cost_mean'], r['quality_mean']),
                textcoords='offset points', xytext=(6, 6), fontsize=9)

ax.set_xlabel('Cost per request (USD)')
ax.set_ylabel('Quality (reward-model score)')
ax.set_title(f"Cost vs Quality — n={MANIFEST['experiment']['n_seeds']} seeds, 95% CI")
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## 3. Per-Router Aggregate Stats

Four panels: average cost, quality, latency, and success rate per router,
with 95% CI error bars. The same numbers feed the comparison table in §4.


In [ ]:
def _stat(router, metric):
    return RAW_STATS[router][metric]['mean'], RAW_STATS[router][metric]['ci']

routers = sorted(RAW_STATS.keys(), key=lambda r: RAW_STATS[r]['cost']['mean'])
metrics = [('cost', 'Cost (USD)'),
           ('quality', 'Quality'),
           ('latency', 'Latency (ms)'),
           ('success', 'Success rate')]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (key, label) in zip(axes.flat, metrics):
    means = [RAW_STATS[r][key]['mean'] for r in routers]
    cis = [RAW_STATS[r][key]['ci'] for r in routers]
    ax.bar(range(len(routers)), means, yerr=cis, capsize=4,
           color='tab:blue', alpha=0.75)
    ax.set_xticks(range(len(routers)))
    ax.set_xticklabels(routers, rotation=30, ha='right')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## 4. Paper-Ready Comparison Table

Mean ± 95% CI for every (router × metric) cell. This table goes directly into
the paper — copy `df.to_latex()` into the manuscript.


In [ ]:
def _fmt(stats, key, prec):
    m = stats[key]['mean']
    c = stats[key]['ci']
    return f"{m:.{prec}f} ± {c:.{prec}f}"

rows = []
for r in routers:
    s = RAW_STATS[r]
    rows.append({
        'Router': r,
        'Cost ($/req)':   _fmt(s, 'cost', 5),
        'Quality':        _fmt(s, 'quality', 4),
        'Latency (ms)':   _fmt(s, 'latency', 0),
        'Success rate':   _fmt(s, 'success', 3),
        'On Pareto':      'yes' if any(
            (PARETO['router'] == r) & (PARETO['on_frontier'] == 1)
        ) else 'no',
    })
df = pd.DataFrame(rows)
df


## 5. Theoretical Predictions

The CD-TS regret analysis in `paper/theory.tex` predicts:

$$
\mathbb{E}[R_T] \;=\; O\!\left((MK \log T)^{1/3} \cdot V_T^{1/3} \cdot T^{2/3}\right)
$$

For the deployment parameters here — $M = 5$ complexity bins, $K \in \{2, 3\}$
engine tiers, and $T$ in the hundreds — the **rate constant matters more than
the asymptotic exponent**: the implied effective-window optimum is

$$
L^\star \;=\; \Theta\!\big((T \sqrt{MK \log T} / V_T)^{2/3}\big)
$$

which for $T = 200$, $MK = 15$, $V_T \approx 5$ gives $L^\star \approx 50$
rounds. The default `decay_factor = 0.995` corresponds to $L = 200$ — slightly
slow for this regime; tuning to $\gamma \approx 0.98$ would track drift more
sharply at the cost of higher stationary variance.

The MDP router has no comparable finite-sample guarantee under our current
$\varepsilon$-greedy parameterization (asymptotic convergence only). For
sample-efficient deployment, CD-TS is the recommended policy.


## 6. Re-running this analysis

To swap the dataset, increase the seed count, or compare system-mode against
pure routing, regenerate the artifact and re-run this notebook:

```bash
# Pure routing comparison (apples-to-apples, fair for the policy contribution):
python -m python_core.scripts.run_experiment --max-samples 500 --seeds 10 \
    --label "paper2-pure-routing"

# Full system stack (cache + privacy + gatekeeper applied to every router):
python -m python_core.scripts.run_experiment --max-samples 500 --seeds 10 \
    --system-mode --label "paper2-system-mode"
```

Both runs produce independent `results/<timestamp>/` directories with their own
manifests. To analyze a specific one, set `RESULTS_DIR` explicitly in cell 1.
